# Phase 2.5 — Empirical timing calibration (n=3 stabilized greedy)

**Purpose:** measure the n=3 solver's **throughput (nodes/sec)** on the hard code path so we can
size the full sweep and pick budget tiers. This is a *timing* run, not a correctness run.

**How to run on Colab**
1. Runtime → Change runtime type → **CPU** (a High-RAM CPU box if offered).
2. Edit **only the CONFIG cell** below (set `REPO_URL`/`BRANCH`, budgets, sample sizes).
3. Runtime → **Run all**. Results stream to `results/calibration.jsonl` (and are copied to Drive).
4. Read the **SUMMARY** cell's table, download `calibration.jsonl`, and share both back.

> Prereq: the branch must contain `greedy_nrel.py`, `stabilize.py`, and this notebook — i.e. they
> must be **committed & pushed** before cloning on Colab.


In [ ]:
# ==================== PHASE 2.5 CALIBRATION — CONFIG (edit ONLY this cell) ====================
# Goal: measure n=3 solver throughput (nodes/sec) to size the full sweep. Not a correctness run.

# --- repo / environment (Colab) ---
REPO_URL          = "https://github.com/Avi161/ACSolverX.git"   # your remote
BRANCH            = "test/stable-ac-moves"
REPO_DIR          = "ACSolverX"        # clone target dir name
CLONE             = True               # False if the repo is already present in this session
MOUNT_DRIVE       = True               # mount Google Drive to persist results across sessions
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/acsolverx_calibration"

# --- what to probe ---
N_GEN                 = 3              # stabilized presentations = 3 generators
ARMS                  = ["r1"]         # z=w arm(s) to time. "r1" = representative content-bearing arm.
                                       #   add "r2" to also time the heaviest arm (longest z-relator, up to 18).
MAX_LEN               = 24             # per-relator length cap (matches env L and the real sweep)
USE_NULL_REVERT_BLOCK = True           # time WITH the null-revert block (matches the real z=w arm)

# --- budgets ---
PROBE_BUDGET      = 200_000            # nodes each HARD probe runs to. Bigger = more honest throughput at
                                       #   depth (it decays as visited/heap grow) but slower. See tuning notes.
CANDIDATE_BUDGETS = [100_000, 1_000_000]   # tiers to PROJECT sweep hours for, from measured nodes/sec

# --- samples ---
HARD_DATASET = "ms_reps_unsolved"      # 261 genuinely-hard reps -> data/ms_unsolved_reps/ms_reps_unsolved.txt
N_HARD       = 5                       # hard probes (drive the estimate; run to PROBE_BUDGET)
HARD_IDX     = None                    # None -> evenly spread; or explicit list e.g. [0, 65, 130, 195, 260]

EASY_DATASET = "1190MS"                # solved presentations -> data/1190MS.txt
N_EASY       = 5                       # easy probes (cheap solved-case time; minor addend)
EASY_IDX     = None                    # None -> spread across the solved region (idx<640); or explicit list
EASY_BUDGET  = 100_000                 # easy cases solve early; cap so a non-solver can't stall the probe

# --- projection: (label, total #presentations, approx #budget-exhausting "unsolved" cases) ---
# sweep hours/arm/tier ~= budget * #unsolved / nodes_per_sec / 3600  (+ small solved-case addend)
PROJECT_OVER = [
    ("MS(1190)",          1190, 550),  # ~550 run to full budget under GS at n=2; n=3 similar-or-more
    ("unsolved reps 261",  261, 261),  # worst case: all 261 exhaust the budget
]
N_ARMS_TOTAL    = 4                    # x,y,r1,r2 -> whole-experiment compute multiplier
WORKERS_PER_BOX = 4                    # cores you'll use per box (divides wall-clock; the probe is 1-core)
# =============================================================================================
print("config loaded.")


In [ ]:
# ==================== SETUP (clone / install / mount / import) ====================
import os, sys, subprocess, ast, time, json, resource, platform, statistics
def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    if CLONE and not os.path.isdir(REPO_DIR):
        sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    REPO_ROOT = os.path.abspath(REPO_DIR)
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while REPO_ROOT != "/" and not os.path.isdir(os.path.join(REPO_ROOT, "envs")):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

ONEGEN = os.path.join(REPO_ROOT, "experiments", "stable_ac", "one_generator")
sys.path.insert(0, ONEGEN)
import greedy_nrel as gn
import stabilize as stab

RESULTS_DIR = os.path.join(ONEGEN, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
CALIB_PATH = os.path.join(RESULTS_DIR, "calibration.jsonl")

RAW = {
    "1190MS":           os.path.join(REPO_ROOT, "data", "1190MS.txt"),
    "ms_reps_unsolved": os.path.join(REPO_ROOT, "data", "ms_unsolved_reps", "ms_reps_unsolved.txt"),
}
def load_raw(name):
    with open(RAW[name]) as f:
        return [ast.literal_eval(l) for l in f if l.strip()]

def peak_rss_mb():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return r / 1024.0 if platform.system() == "Linux" else r / (1024.0 * 1024.0)  # Linux KB, macOS bytes

print("repo :", REPO_ROOT)
print("out  :", CALIB_PATH)


In [ ]:
# ==================== WARM UP numba (so JIT-compile time is not inside a measurement) ====================
_warm = stab.stabilize_flat(load_raw("1190MS")[0], "r1")
t0 = time.time()
gn.solve_one(_warm, n_gen=N_GEN, max_len=MAX_LEN, max_nodes=2000)
print(f"warmup (numba JIT of all primitives) done in {time.time()-t0:.1f}s")


In [ ]:
# ==================== BUILD PROBE INDICES ====================
def spread_idx(n_total, k):
    if k >= n_total: return list(range(n_total))
    return sorted({round(i * (n_total - 1) / (k - 1)) for i in range(k)})

hard_raw = load_raw(HARD_DATASET)
easy_raw = load_raw(EASY_DATASET)
hard_idx = HARD_IDX if HARD_IDX is not None else spread_idx(len(hard_raw), N_HARD)
easy_idx = EASY_IDX if EASY_IDX is not None else spread_idx(640, N_EASY)  # 1190MS solved region is idx<640
print(f"HARD ({HARD_DATASET}, {len(hard_raw)} lines) idx:", hard_idx)
print(f"EASY ({EASY_DATASET}) idx:", easy_idx)


In [ ]:
# ==================== RUN PROBES (resumable append-only JSONL) ====================
def run_probe(kind, dataset, raw, idx, arm, budget):
    sflat   = stab.stabilize_flat(raw[idx], arm)                       # n=2 -> n=3 (z=w)
    blocked = [gn.null_revert_state(sflat, N_GEN)] if USE_NULL_REVERT_BLOCK else None
    res, _  = gn.solve_one(sflat, n_gen=N_GEN, max_len=MAX_LEN, max_nodes=budget, blocked_states=blocked)
    nps = res["nodes_explored"] / res["wall_time_s"] if res["wall_time_s"] > 0 else 0
    rec = {"kind": kind, "dataset": dataset, "idx": idx, "arm": arm, "n_gen": N_GEN,
           "budget_nodes": budget, "nodes_explored": res["nodes_explored"], "solved": res["solved"],
           "path_verified": res["path_verified"], "path_len": res["path_len"],
           "revert_hits": res["revert_hits"], "wall_time_s": res["wall_time_s"],
           "nodes_per_sec": round(nps), "peak_rss_mb": round(peak_rss_mb(), 1),
           "exhausted_budget": (not res["solved"])}
    with open(CALIB_PATH, "a") as f:
        f.write(json.dumps(rec) + "\n"); f.flush(); os.fsync(f.fileno())
    print(f"  [{kind:4}] {dataset:16} idx {idx:4} z={arm:2}: solved={rec['solved']!s:5} "
          f"nodes={rec['nodes_explored']:8} {rec['wall_time_s']:7.1f}s  {rec['nodes_per_sec']:6}/s  "
          f"rss={rec['peak_rss_mb']:.0f}MB  reverts={rec['revert_hits']}")
    return rec

done = set()
if os.path.exists(CALIB_PATH):
    for l in open(CALIB_PATH):
        try:
            r = json.loads(l); done.add((r["kind"], r["dataset"], r["idx"], r["arm"], r["budget_nodes"]))
        except Exception:
            pass

for arm in ARMS:
    print(f"=== ARM z={arm} ===")
    print(" HARD probes (run to PROBE_BUDGET):")
    for idx in hard_idx:
        if ("hard", HARD_DATASET, idx, arm, PROBE_BUDGET) in done: print(f"  skip hard idx {idx}"); continue
        run_probe("hard", HARD_DATASET, hard_raw, idx, arm, PROBE_BUDGET)
    print(" EASY probes:")
    for idx in easy_idx:
        if ("easy", EASY_DATASET, idx, arm, EASY_BUDGET) in done: print(f"  skip easy idx {idx}"); continue
        run_probe("easy", EASY_DATASET, easy_raw, idx, arm, EASY_BUDGET)

if IN_COLAB and MOUNT_DRIVE:
    import shutil; shutil.copy(CALIB_PATH, os.path.join(DRIVE_RESULTS_DIR, "calibration.jsonl"))
    print("copied results ->", DRIVE_RESULTS_DIR)
print("done.")


In [ ]:
# ==================== SUMMARY & SWEEP PROJECTION ====================
recs = [json.loads(l) for l in open(CALIB_PATH) if l.strip()]
hard = [r for r in recs if r["kind"] == "hard"]
easy = [r for r in recs if r["kind"] == "easy"]
print(f"probes: {len(hard)} hard, {len(easy)} easy   (arms: {sorted(set(r['arm'] for r in recs))})\n")

exh     = [r["nodes_per_sec"] for r in hard if r["exhausted_budget"]]
allhard = [r["nodes_per_sec"] for r in hard]
nps_est = statistics.median(exh) if exh else (statistics.median(allhard) if allhard else 0)

print(f"THROUGHPUT (nodes/sec, 1 core):")
if allhard:
    print(f"  hard all      : median {statistics.median(allhard):.0f}  min {min(allhard):.0f}  max {max(allhard):.0f}")
print(f"  hard exhausting: median {nps_est:.0f}   <-- used for the projection ({len(exh)}/{len(hard)} exhausted budget)")
if not exh:
    print("  WARNING: no hard probe ran to full budget -> this nodes/sec is SOLVE-time throughput (OPTIMISTIC).")
    print("           raise PROBE_BUDGET or choose harder HARD_IDX so probes exhaust the budget.")
if easy:
    print(f"  easy solved   : median wall {statistics.median([r['wall_time_s'] for r in easy]):.2f}s (small addend)")
print(f"  peak RSS      : {max((r['peak_rss_mb'] for r in recs), default=0):.0f} MB (x WORKERS_PER_BOX on a multiprocess box)\n")

if nps_est > 0:
    print(f"PROJECTED SWEEP TIME  (per arm; fleet runs one arm per box in parallel):")
    hdr = f"  {'dataset':18} {'#unsolved':>9} {'tier':>11} {'hrs @1core':>11} {'hrs @'+str(WORKERS_PER_BOX)+'core':>11}"
    print(hdr); print("  " + "-" * (len(hdr) - 2))
    for label, ntot, nuns in PROJECT_OVER:
        for tier in CANDIDATE_BUDGETS:
            hrs1 = tier * nuns / nps_est / 3600.0
            hrsW = hrs1 / max(1, WORKERS_PER_BOX)
            print(f"  {label:18} {nuns:9} {tier:11,} {hrs1:11.1f} {hrsW:11.1f}")
    print()
    print("  Reading it: each row = ONE z=w arm on ONE box. With one-arm-per-box the FLEET wall-clock")
    print(f"  ~= the largest single-arm 'hrs @{WORKERS_PER_BOX}core' (arms run in parallel). Whole-experiment")
    print(f"  COMPUTE = that x {N_ARMS_TOTAL} arms. Add the easy-tail addend (small). Throughput decays with")
    print("  depth, so a PROBE_BUDGET below your target tier makes the large-tier estimate optimistic.")
